In [16]:
import pandas as pd

sales = pd.read_csv("data/king_county_house_sales.csv")
details = pd.read_csv("data/king_county_house_details.csv")

# Left Join: alle Verkäufe behalten, Hausdetails dazuholen
df = sales.merge(
    details,
    left_on="house_id",   # Schlüssel in sales -> verweist auf das Haus
    right_on="id",        # Schlüssel in details
    how="left",
    suffixes=("", "_details"),
)

print(f"combined: {df.shape[0]} rows, {df.shape[1]} columns")
assert len(df) == len(sales), "Es sollte genau eine Zeile pro Verkauf geben"
df.head()
df.shape

combined: 21597 rows, 23 columns


(21597, 23)

In [17]:
print(f"{'Quelle':<12} {'Zeilen':>8} {'Spalten':>8}")
print(f"{'sales':<12} {len(sales):>8} {sales.shape[1]:>8}")
print(f"{'details':<12} {len(details):>8} {details.shape[1]:>8}")
print(f"{'merged (df)':<12} {len(df):>8} {df.shape[1]:>8}")

Quelle         Zeilen  Spalten
sales           21597        4
details         21420       19
merged (df)     21597       23


In [18]:
# Eindeutigkeit der Schlüssel in der zusammengeführten Tabelle
check = pd.DataFrame({
    "rows": len(df),
    "unique": df[["id", "house_id"]].nunique(),
    "is_unique": [df["id"].is_unique, df["house_id"].is_unique],
})
check["duplicates"] = check["rows"] - check["unique"]
print(check, "\n")

# Verkaufs-ID muss eindeutig sein: eine Zeile pro Verkauf
assert df["id"].is_unique, "Verkaufs-id ist nicht eindeutig: Der Merge hat Zeilen vervielfacht"

# house_id darf mehrfach vorkommen: Häuser, die mehrmals verkauft wurden
sales_per_house = df["house_id"].value_counts()
print("Häuser mit mehreren Verkäufen:", (sales_per_house > 1).sum())
print(sales_per_house.value_counts().sort_index().rename_axis("Verkäufe pro Haus").rename("Anzahl Häuser"), "\n")

# Beispiel: alle Verkäufe des am häufigsten verkauften Hauses
top_house = sales_per_house.index[0]
df.loc[df["house_id"] == top_house, ["id", "house_id", "date", "price"]]


           rows  unique  is_unique  duplicates
id        21597   21597       True           0
house_id  21597   21420      False         177 

Häuser mit mehreren Verkäufen: 176
Verkäufe pro Haus
1    21244
2      175
3        1
Name: Anzahl Häuser, dtype: int64 



,id,house_id,date,price
17588,17589,795000620,2014-09-24,115000.0
17589,17590,795000620,2014-12-15,124000.0
17590,17591,795000620,2015-03-11,157000.0


## 1. Pflicht-Audit der Daten

Verkürzte EDA nach Abschnitt 0, Schritt 1 der Entscheidungsmatrix (`sklearn_decision_matrix_EN.md`): Typen, fehlende Werte, ungültige Einträge, Duplikate, Wertebereiche und Verteilung des Targets. Jeder Befund wird über die Matrix in Abschnitt 1 einer Preprocessing-Entscheidung zugeordnet.

**Reihenfolge:**
- **Vor dem Split, auf `df`:** 1a, 1c, 1d. Diese Schritte prüfen oder ändern jede Zeile nach einer festen Regel und lernen nichts aus der Verteilung der Daten.
- **Nach dem Split, nur auf `df_train`:** 1b, 1e, 1f. Aus diesen Analysen werden Entscheidungen abgeleitet (Imputation, Skalierung, Log-Transformation). Deshalb dürfen sie das Testset nicht sehen.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1a. Datentypen

Hat jede Spalte den passenden Typ? `date` wird als Text eingelesen und deshalb in einen Zeitstempel umgewandelt. `zipcode` ist numerisch gespeichert, inhaltlich aber eine **nominale Kategorie**, also später ein Fall für `OneHotEncoder`.

In [4]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

date             datetime64[us]
price                   float64
house_id                  int64
id                        int64
id_details                int64
bedrooms                float64
bathrooms               float64
sqft_living             float64
sqft_lot                float64
floors                  float64
waterfront              float64
view                    float64
condition                 int64
grade                     int64
sqft_above              float64
sqft_basement           float64
yr_built                  int64
yr_renovated            float64
zipcode                   int64
lat                     float64
long                    float64
sqft_living15           float64
sqft_lot15              float64
dtype: object

## 1c. Ungültige Einträge und Sondermarker

- Welche Werte kommen in den diskreten Spalten vor? Gibt es unplausible Codes?
- `yr_renovated == 0` bedeutet vermutlich „nie renoviert“. Das ist ein **Sondermarker**, kein Jahr. Matrix: in NaN umwandeln oder als eigenes Feature kodieren.
- Konsistenz: Es sollte `sqft_living == sqft_above + sqft_basement` gelten.
- Gibt es Häuser ohne Schlaf- oder Badezimmer?

In [6]:
for col in ["bedrooms", "bathrooms", "floors", "waterfront", "view", "condition", "grade"]:
    print(f"{col:<11}", sorted(df[col].dropna().unique()))

print("\nyr_renovated == 0:", (df["yr_renovated"] == 0).sum(), "Zeilen")

both = df[["sqft_above", "sqft_basement"]].notna().all(axis=1)
mismatch = df.loc[both, "sqft_living"] != df.loc[both, "sqft_above"] + df.loc[both, "sqft_basement"]
print("sqft_living != sqft_above + sqft_basement:", mismatch.sum(), "Zeilen")

print("0 Schlaf- oder Badezimmer:", ((df["bedrooms"] == 0) | (df["bathrooms"] == 0)).sum(), "Zeilen")

bedrooms    [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0)]
bathrooms   [np.float64(0.5), np.float64(0.75), np.float64(1.0), np.float64(1.25), np.float64(1.5), np.float64(1.75), np.float64(2.0), np.float64(2.25), np.float64(2.5), np.float64(2.75), np.float64(3.0), np.float64(3.25), np.float64(3.5), np.float64(3.75), np.float64(4.0), np.float64(4.25), np.float64(4.5), np.float64(4.75), np.float64(5.0), np.float64(5.25), np.float64(5.5), np.float64(5.75), np.float64(6.0), np.float64(6.25), np.float64(6.5), np.float64(6.75), np.float64(7.5), np.float64(7.75), np.float64(8.0)]
floors      [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5)]
waterfront  [np.float64(0.0), np.float64(1.0)]
view        [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
condition   [np.in

## 1d. Duplikate

- **Exakte Duplikate**, also Zeilen, die bis auf die Verkaufs-`id` identisch sind, wären Erfassungsfehler und werden vor dem Split entfernt.
- **Mehrfachverkäufe** desselben Hauses sind echte Daten. Sie sind trotzdem wichtig für den Split, denn sonst kann dasselbe Haus sowohl in Train als auch in Test landen.

In [ ]:
print("Exakte Duplikate (ohne Verkaufs-id):", df.drop(columns="id").duplicated().sum())

# Fehlerhafte Duplikate vor dem Split entfernen
df = df.drop_duplicates(subset=df.columns.drop("id"))

sales_per_house = df["house_id"].value_counts()
print("Häuser mit mehreren Verkäufen:", (sales_per_house > 1).sum())
print("\nAnzahl Verkäufe pro Haus:")
print(sales_per_house.value_counts().sort_index())

## Train/Test-Split (gruppiert nach Haus)

Der Split erfolgt, **bevor** aus den Verteilungen der Daten Entscheidungen abgeleitet werden. Ab hier laufen die Analysen, die Feature-Auswahl und das Tuning nur auf `df_train`. `df_test` bleibt bis zur finalen Evaluation unberührt.

Weil einige Häuser mehrmals verkauft wurden, wird nach `house_id` **gruppiert** geteilt: Alle Verkäufe eines Hauses landen entweder im Train- oder im Testset. `test_size=0.2` bezieht sich daher auf 20 % der Häuser, nicht der Zeilen. Später in der Cross-Validation aus demselben Grund `GroupKFold` mit `groups=df_train["house_id"]` verwenden.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["house_id"]))
df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

print(f"train: {len(df_train)} rows ({len(df_train) / len(df):.1%}), test: {len(df_test)} rows ({len(df_test) / len(df):.1%})")
n_shared = df_train["house_id"].isin(df_test["house_id"]).sum()
print("Häuser in Train und Test:", n_shared)
assert n_shared == 0, "Ein Haus kommt in Train und Test vor"

## 1b. Fehlende Werte

Anzahl und Anteil der fehlenden Werte pro Spalte. Matrix: numerische Lücken → `SimpleImputer(strategy="median")`, kategoriale Lücken → `SimpleImputer(strategy="most_frequent")`.

In [ ]:
missing = pd.DataFrame({
    "n_missing": df_train.isna().sum(),
    "pct_missing": (df_train.isna().mean() * 100).round(2),
})
missing[missing["n_missing"] > 0].sort_values("n_missing", ascending=False)

## 1e. Wertebereiche

Minimum, Maximum und Quantile jeder numerischen Spalte, ohne die IDs. Achte auf unplausible Extremwerte und stark unterschiedliche Skalen. Matrix: unterschiedliche Skalen → `StandardScaler`, Extremwerte → untersuchen und `RobustScaler` vergleichen.

In [ ]:
df_train.drop(columns=["id", "house_id", "id_details"]).describe().T.round(2)

## 1f. Verteilung des Targets

Wie ist `price` verteilt? Ist der Preis stark rechtsschief, lohnt sich später ein Vergleich mit einem Modell auf `log(price)`. Matrix: rechtsschief → `np.log1p`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_train["price"], bins=60, ax=axes[0])
axes[0].set_title(f"price (skew = {df_train['price'].skew():.2f})")
sns.histplot(np.log1p(df_train["price"]), bins=60, ax=axes[1])
axes[1].set_title(f"log1p(price) (skew = {np.log1p(df_train['price']).skew():.2f})")
plt.tight_layout()
plt.show()

df_train["price"].describe()

## Features und Target trennen

`y_train` ist der Preis, also die Größe, die vorhergesagt wird. `X_train` enthält alle übrigen Spalten. Das Testset bleibt unberührt, bis es am Ende genauso vorbereitet und in `X_test` und `y_test` geteilt wird.

In [ ]:
TARGET = "price"

X_train = df_train.drop(columns=TARGET)
y_train = df_train[TARGET]

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")